# Standing Shoulder Abduction - TCN

Este notebook usa `Standing_Shoulder_Abduction.py` para:
- Entrada: ventana temporal de frames desde JSON (`jsonEjemplo.json`)
- Features biomecanicas (angulo hombro-brazo, altura relativa, elevacion escapular, tronco, simetria, velocidad angular, rango maximo)
- Modelo: TCN
- Salidas: correcta/incorrecta, severidad y errores detectados
- Objetivo principal: maximizar sensibilidad/recall de ejecuciones incorrectas

## 0. Dependencias
Si usas Python 3.14, TensorFlow puede no estar disponible.
Para entrenar TCN, usa preferiblemente Python 3.11 o 3.12.

In [48]:
# Ejecuta solo si necesitas instalar en este kernel
%pip install -q numpy pandas scikit-learn joblib tensorflow

Note: you may need to restart the kernel to use updated packages.


## 1. Importaciones

In [49]:
from pathlib import Path
import json
import importlib
import numpy as np
import joblib

import Standing_Shoulder_Abduction as ssa
ssa = importlib.reload(ssa)

print(f'BASE_DIR: {Path.cwd()}')
print(f'JSON ejemplo: {ssa.JSON_EXAMPLE_PATH}')
print(f'Dataset real esperado: {ssa.DATASET_ROOT}')
print(f'Artifacts: {ssa.ARTIFACTS_DIR}')

BASE_DIR: c:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_abduction
JSON ejemplo: C:\Users\marco\notebooks\modelo_rehabilitacion\jsonEjemplo.json
Dataset real esperado: C:\Users\marco\notebooks\modelo_rehabilitacion\dataset
Artifacts: C:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_abduction\standing_shoulder_abduction_artifacts


## 2. Vista rapida del JSON de entrada

In [50]:
payload = ssa.load_json_payload(ssa.JSON_EXAMPLE_PATH)

print('Claves de primer nivel:', list(payload.keys()))
frames = ssa.coerce_frames(payload)
print('Cantidad de frames detectados:', len(frames))

if frames:
    joints = list(frames[0]['puntos_clave'].keys())
    print('Joints disponibles en el primer frame:', joints)
    if 't' in frames[0]:
        print("Primer timestamp 't':", frames[0]['t'])

Claves de primer nivel: ['ejercicio_id', 'secuencia']
Cantidad de frames detectados: 100
Joints disponibles en el primer frame: ['nariz', 'hombro_i', 'hombro_d', 'codo_d', 'muneca_d']
Primer timestamp 't': 0


## 3. Construccion de dataset

Usa tu dataset real UI-PRMD desde la carpeta `dataset`:

- `dataset/Segmented Movements/Segmented Movements/Kinect/Positions`

- `dataset/Incorrect Segmented Movements/Incorrect Segmented Movements/Kinect/Positions`

- `dataset/Movements/Movements/Kinect/Positions`

- `dataset/Incorrect Movements/Incorrect Movements/Kinect/Positions`

El entrenamiento queda forzado a UI-PRMD (m09) y no usa fallback a carpetas JSON/sinteticas.

In [51]:
X, y, meta, dataset_source = ssa.get_dataset()
print(f'dataset_source: {dataset_source}')
print(f'X shape: {X.shape}')
print(f'y balance: {np.bincount(y)}')

coverage = meta.attrs.get('coverage', {})
if coverage:
    print('\nCobertura loader m09:')
    for k, v in coverage.items():
        print(f'  {k}: {v}')

if 'folder_split' in meta.columns:
    print('\nDistribucion por carpeta origen:')
    print(meta.groupby(['folder_split', 'label']).size().unstack(fill_value=0))

meta.head()

dataset_source: ui_prmd_kinect_positions
X shape: (220, 48, 10)
y balance: [110 110]

Cobertura loader m09:
  candidate_files: 220
  used_sequences: 220
  skipped_load: 0
  skipped_shape: 0
  skipped_features: 0

Distribucion por carpeta origen:
label                            0    1
folder_split                           
Incorrect Movements              0   10
Incorrect Segmented Movements    0  100
Movements                       10    0
Segmented Movements            100    0


,sequence_id,subject_id,label,file,folder_split,is_segmented,n_frames_raw,max_shoulder_angle_deg,mean_shoulder_angle_deg,trunk_tilt_p95_deg,shoulder_elevation_p95,symmetry_p95_deg,velocity_std_deg,wrist_above_elbow_mean,valid_points_ratio_mean
0,m09_s01_e01_positions,1,0,m09_s01_e01_positions.txt,Segmented Movements,True,77,98.805626,98.471611,172.042435,0.007452,74.084755,0.041379,0.004078,1.0
1,m09_s01_e02_positions,1,0,m09_s01_e02_positions.txt,Segmented Movements,True,80,98.080338,97.920410,172.240021,0.001604,74.479958,0.019783,0.005479,1.0
2,m09_s01_e03_positions,1,0,m09_s01_e03_positions.txt,Segmented Movements,True,81,98.516258,98.129478,172.366074,0.012119,74.732018,0.021434,0.005808,1.0
3,m09_s01_e04_positions,1,0,m09_s01_e04_positions.txt,Segmented Movements,True,83,99.604294,99.179832,171.439072,0.004645,72.878067,0.052322,0.005857,1.0
4,m09_s01_e05_positions,1,0,m09_s01_e05_positions.txt,Segmented Movements,True,77,99.088486,98.916992,171.413742,0.015893,72.827370,0.018629,0.006351,1.0


## 4. Entrenamiento del TCN (foco en recall)
Esta celda entrena, ajusta umbral para recall objetivo y exporta artifacts.

In [52]:
results = None
try:
    results = ssa.train_and_export()
    print('Entrenamiento completado.')
    print('Metricas principales:')
    for k, v in results['metrics'].items():
        print(f'  {k}: {v}')
except RuntimeError as exc:
    print(str(exc))
    print('Sugerencia: usa un entorno Python 3.11/3.12 para TensorFlow y vuelve a ejecutar.')

=== Standing Shoulder Abduction / TCN ===
Dataset usado        : ui_prmd_kinect_positions
Muestras totales     : 220
Threshold optimizado : 0.4740
Recall incorrecto    : 0.9259
Precision incorrecto : 0.6579
F1 incorrecto        : 0.7692
ROC-AUC              : 0.8810
Matriz de confusión:
[[15 13]
 [ 2 25]]
Reporte de clasificación:
              precision    recall  f1-score   support

    correcta     0.8824    0.5357    0.6667        28
  incorrecta     0.6579    0.9259    0.7692        27

    accuracy                         0.7273        55
   macro avg     0.7701    0.7308    0.7179        55
weighted avg     0.7722    0.7273    0.7170        55

Demo de inferencia:
{
  "exercise": "Standing Shoulder Abduction",
  "movement_id": 9,
  "label": 1,
  "classification": "incorrecta",
  "probability_incorrect": 1.0,
  "threshold": 0.474,
  "severity": "severa",
  "errors_detected": [
    "inclinacion_excesiva_tronco",
    "elevacion_escapular_excesiva",
    "asimetria_derecha_izquierda"

## 5. Inferencia demo
Salida esperada:
- etiqueta: correcta / incorrecta
- severidad
- errores_detectados

In [53]:
try:
    tf = ssa.require_tensorflow()
    bundle_path = ssa.ARTIFACTS_DIR / 'standing_shoulder_abduction_bundle.joblib'
    model_path = ssa.ARTIFACTS_DIR / 'standing_shoulder_abduction_tcn.keras'

    if not bundle_path.exists() or not model_path.exists():
        raise FileNotFoundError('No existen artifacts aun. Ejecuta la celda de entrenamiento primero.')

    bundle = joblib.load(bundle_path)
    model = tf.keras.models.load_model(model_path)

    demo_payload = ssa.load_json_payload(ssa.JSON_EXAMPLE_PATH)
    demo_frames, _ = ssa.generate_temporal_window_from_example(
        payload=demo_payload,
        rng=np.random.default_rng(123),
        incorrect=True,
        n_frames=ssa.WINDOW_SIZE,
    )

    pred = ssa.infer_from_window(demo_frames, model=model, bundle=bundle)
    print(json.dumps(pred, ensure_ascii=False, indent=2))
except Exception as exc:
    print(f'No se pudo ejecutar inferencia: {exc}')

{
  "exercise": "Standing Shoulder Abduction",
  "movement_id": 9,
  "label": 1,
  "classification": "incorrecta",
  "probability_incorrect": 1.0,
  "threshold": 0.474,
  "severity": "severa",
  "errors_detected": [
    "inclinacion_excesiva_tronco",
    "elevacion_escapular_excesiva",
    "asimetria_derecha_izquierda",
    "velocidad_angular_irregular"
  ],
  "biomechanical_summary": {
    "max_shoulder_angle_deg": 179.3995,
    "mean_shoulder_angle_deg": 158.3373,
    "trunk_tilt_p95_deg": 148.983,
    "shoulder_elevation_p95": 0.3631,
    "symmetry_p95_deg": 18.8056,
    "velocity_std_deg": 5.1112,
    "wrist_above_elbow_mean": 0.3468,
    "valid_points_ratio_mean": 1.0
  }
}


## 6. Uso en produccion (resumen)

1. Carga artifacts exportados (`.joblib` + `.keras`).
2. Recibe payload JSON con `frames`, `secuencia`, `puntos` o `puntos_clave`.
3. Normaliza a secuencia con `ssa.coerce_frames(...)`.
4. Llama `ssa.infer_from_window(...)` y devuelve:
   - `etiqueta` (`correcta` / `incorrecta`)
   - `severidad`
   - `errores_detectados`
   - `probabilidad_incorrecta`
   - `resumen_biomecanico`

Ejemplo minimo de carga:

```python
from pathlib import Path
import joblib
import tensorflow as tf
import Standing_Shoulder_Abduction as ssa

artifacts = Path('standing_shoulder_abduction_artifacts')
bundle = joblib.load(artifacts / 'standing_shoulder_abduction_bundle.joblib')
model = tf.keras.models.load_model(artifacts / 'standing_shoulder_abduction_tcn.keras')
```

Ejemplo minimo de prediccion:

```python
def predecir_abduccion(payload_json: dict) -> dict:
    frames = ssa.coerce_frames(payload_json)
    if len(frames) < 2:
        frames = frames * max(2, ssa.WINDOW_SIZE)
    return ssa.infer_from_window(frames_json=frames, model=model, bundle=bundle)
```

Nota: para entrenar TCN usa Python 3.11 o 3.12.

In [54]:
from pathlib import Path
import joblib
import tensorflow as tf
import Standing_Shoulder_Abduction as ssa

artifacts = Path('standing_shoulder_abduction_artifacts')
bundle = joblib.load(artifacts / 'standing_shoulder_abduction_bundle.joblib')
model = tf.keras.models.load_model(artifacts / 'standing_shoulder_abduction_tcn.keras')

def predecir_abduccion(payload_json: dict) -> dict:
    frames = ssa.coerce_frames(payload_json)
    if len(frames) < 2:
        frames = frames * max(2, ssa.WINDOW_SIZE)
    return ssa.infer_from_window(frames_json=frames, model=model, bundle=bundle)




In [55]:
predecir_abduccion({
    "ejercicio_id": "m09",
    "secuencia": [
        {
            "puntos": {
                "nariz": {"x": 0.50, "y": 0.20, "z": -0.15},
                "hombro_d": {"x": 0.40, "y": 0.35, "z": -0.05},
                "codo_d": {"x": 0.35, "y": 0.50, "z": -0.08},
                "muneca_d": {"x": 0.30, "y": 0.65, "z": -0.12}
            },
            "t": 0
        },
        {
            "puntos": {
                "nariz": {"x": 0.50, "y": 0.20, "z": -0.15},
                "hombro_d": {"x": 0.40, "y": 0.34, "z": -0.05},
                "codo_d": {"x": 0.33, "y": 0.45, "z": -0.08},
                "muneca_d": {"x": 0.28, "y": 0.58, "z": -0.12}
            },
            "t": 33
        }
    ]
})

{'exercise': 'Standing Shoulder Abduction',
 'movement_id': 9,
 'label': 1,
 'classification': 'incorrecta',
 'probability_incorrect': 1.0,
 'threshold': 0.474,
 'severity': 'severa',
 'errors_detected': ['inclinacion_excesiva_tronco',
  'descenso_de_codo_o_muneca'],
 'biomechanical_summary': {'max_shoulder_angle_deg': 176.9335,
  'mean_shoulder_angle_deg': 170.8392,
  'trunk_tilt_p95_deg': 35.4453,
  'shoulder_elevation_p95': 0.0003,
  'symmetry_p95_deg': 0.0,
  'velocity_std_deg': 0.0,
  'wrist_above_elbow_mean': -0.4782,
  'valid_points_ratio_mean': 0.6667}}